# 02 — Clean Meter Inventory

I use the LADOT meter inventory to prepare parking characteristics for the cross-sectional analysis. I first inspect the raw records, then remove exact duplicates, construct numeric variables, and check matches to the occupancy sample from Notebook 01.


## 1. Load the raw meter inventory

I start with the raw LADOT meter inventory and inspect a few rows before making any changes. The source file stays unchanged, and I work with a separate in-memory copy.


In [11]:
import pandas as pd

meters = pd.read_csv(
    "../../data/raw/ladot_meters/20261017T052111914317Z/inventory.csv"
)

meters.head()

,SpaceID,BlockFace,MeterType,RateType,RateRange,MeteredTimeLimit,LatLng
0,CB3201,201 8TH ST,Single-Space,TOD,$5.00 - $7.00,2HR,"(34.04163, -118.251873)"
1,WV156,10900 LE CONTE AVE,Single-Space,FLAT,$3.00,2HR,"(34.063573, -118.447284)"
2,WV250,10900 LE CONTE AVE,Single-Space,FLAT,$3.00,2HR,"(34.063589, -118.446051)"
3,SH248,1000 HIGHLAND AVE,Single-Space,FLAT,$1.50,1HR,"(34.089736, -118.338477)"
4,SH5,6201 SANTA MONICA BLVD,Single-Space,FLAT,$1.50,1HR,"(34.090827, -118.325045)"


## 2. Check the inventory structure and categories

I check the dimensions, data types, missing values, and number of unique SpaceIDs, then inspect the meter types, pricing categories, rate strings, and time-limit labels. The file has 35,022 rows but 35,015 unique IDs, with no missing values in the raw columns. These are inspection steps; the value counts do not remove or recode any observations.


In [12]:
print("Rows and columns:", meters.shape)

print("\nColumn types:")
print(meters.dtypes)

print("\nMissing values:")
print(meters.isnull().sum())

print("\nUnique SpaceIDs:", meters["SpaceID"].nunique())
print("Duplicate SpaceIDs:", meters["SpaceID"].duplicated().sum())

Rows and columns: (35022, 7)

Column types:
SpaceID             object
BlockFace           object
MeterType           object
RateType            object
RateRange           object
MeteredTimeLimit    object
LatLng              object
dtype: object

Missing values:
SpaceID             0
BlockFace           0
MeterType           0
RateType            0
RateRange           0
MeteredTimeLimit    0
LatLng              0
dtype: int64

Unique SpaceIDs: 35015
Duplicate SpaceIDs: 7


In [13]:
print("Meter types:")
print(meters["MeterType"].value_counts())

print("\nRate types:")
print(meters["RateType"].value_counts())

print("\nRate ranges:")
print(meters["RateRange"].value_counts().head(15))

print("\nTime limits:")
print(meters["MeteredTimeLimit"].value_counts())

Meter types:
MeterType
Single-Space    33769
Multi-Space      1253
Name: count, dtype: int64

Rate types:
RateType
FLAT    27433
TOD      6080
JUMP     1509
Name: count, dtype: int64

Rate ranges:
RateRange
$1.50              24532
$1.00               1318
$1.5/H - $6/10H      799
$0.50 - $1.00        772
$0.50                698
$0.50 - $2.00        496
$0.50 - $3.00        435
$2.50                432
$0.50 - $1.50        419
$1.00 - $2.00        388
$1.00 - $3.00        378
$1.00 - $1.50        359
$1.50 - $3.00        294
$0.50 - $5.00        293
$1/H - $4/10H        260
Name: count, dtype: int64

Time limits:
MeteredTimeLimit
2HR          22453
1HR           6886
4HR           3616
10HR          1094
30MIN          314
6HR            240
15MIN          226
14HR            90
13HR            81
2HR-30MIN       11
7HR             11
Name: count, dtype: int64


## 3. Confirm and remove exact duplicate records

I inspect both copies of the seven repeated SpaceIDs to confirm that their recorded attributes are identical. I then remove exact duplicate rows, rather than arbitrarily choosing between conflicting records with the same ID. The result is 35,015 rows and 35,015 unique SpaceIDs, establishing one inventory row per parking space.


In [14]:
duplicate_ids = meters[meters["SpaceID"].duplicated(keep=False)]

duplicate_ids.sort_values("SpaceID")

,SpaceID,BlockFace,MeterType,RateType,RateRange,MeteredTimeLimit,LatLng
10842,CT252,700 HILL ST,Single-Space,TOD,$0.50 - $3.00,2HR,"(34.06157, -118.240023)"
14506,CT252,700 HILL ST,Single-Space,TOD,$0.50 - $3.00,2HR,"(34.06157, -118.240023)"
5463,CT253,700 HILL ST,Single-Space,TOD,$0.50 - $3.00,2HR,"(34.061637, -118.23999)"
19874,CT253,700 HILL ST,Single-Space,TOD,$0.50 - $3.00,2HR,"(34.061637, -118.23999)"
12531,CT254,700 HILL ST,Single-Space,TOD,$0.50 - $3.00,2HR,"(34.06169, -118.239964)"
19424,CT254,700 HILL ST,Single-Space,TOD,$0.50 - $3.00,2HR,"(34.06169, -118.239964)"
6122,CT256,700 HILL ST,Single-Space,TOD,$0.50 - $3.00,2HR,"(34.061857, -118.239878)"
24465,CT256,700 HILL ST,Single-Space,TOD,$0.50 - $3.00,2HR,"(34.061857, -118.239878)"
11362,CT257,700 HILL ST,Single-Space,TOD,$0.50 - $3.00,2HR,"(34.062011, -118.239799)"
16811,CT257,700 HILL ST,Single-Space,TOD,$0.50 - $3.00,2HR,"(34.062011, -118.239799)"


In [15]:
meters = meters.drop_duplicates().copy()

print("Rows after removing duplicates:", len(meters))
print("Unique SpaceIDs:", meters["SpaceID"].nunique())
print("Duplicate SpaceIDs:", meters["SpaceID"].duplicated().sum())

Rows after removing duplicates: 35015
Unique SpaceIDs: 35015
Duplicate SpaceIDs: 0


## 4. Express parking time limits in hours

I inspect the remaining time-limit labels and map them to numeric `time_limit_hours` for later analysis. The existing mapping treats `2HR-30MIN` as 2.5 hours; this is an interpretation of the compound label, not a separately verified parking rule. I keep `MeteredTimeLimit` alongside the numeric version so the original information remains available.


In [16]:
meters["MeteredTimeLimit"].value_counts().sort_index()

MeteredTimeLimit
10HR          1094
13HR            81
14HR            90
15MIN          226
1HR           6886
2HR          22446
2HR-30MIN       11
30MIN          314
4HR           3616
6HR            240
7HR             11
Name: count, dtype: int64

In [17]:
time_map = {
    "15MIN": 0.25,
    "30MIN": 0.5,
    "1HR": 1,
    "2HR": 2,
    "2HR-30MIN": 2.5,
    "4HR": 4,
    "6HR": 6,
    "7HR": 7,
    "10HR": 10,
    "13HR": 13,
    "14HR": 14
}

meters["time_limit_hours"] = meters["MeteredTimeLimit"].map(time_map)

meters["time_limit_hours"].describe()

count    35015.000000
mean         2.320520
std          1.820516
min          0.250000
25%          2.000000
50%          2.000000
75%          2.000000
max         14.000000
Name: time_limit_hours, dtype: float64

## 5. Inspect pricing structures and extract dollar amounts

I inspect rate strings separately for FLAT, time-of-day (TOD), and JUMP meters because their pricing structures differ. I then extract only the dollar-denominated amounts into `price_numbers`, leaving duration labels such as `/10H` out of the extraction. This is a parsing step, not a filter or a conversion of multi-hour charges into hourly rates.


In [18]:
for rate_type in ["FLAT", "TOD", "JUMP"]:
    print("\n", rate_type)
    print(
        meters[meters["RateType"] == rate_type]["RateRange"]
        .value_counts()
        .head(15)
    )


 FLAT
RateRange
$1.50    24532
$1.00     1318
$0.50      698
$2.50      432
$2.00      221
$3.00       83
$7.00       56
$5.00       42
$8.00       23
$4.00       20
$6.00        8
Name: count, dtype: int64

 TOD
RateRange
$0.50 - $1.00    772
$0.50 - $2.00    496
$0.50 - $3.00    428
$0.50 - $1.50    419
$1.00 - $2.00    388
$1.00 - $3.00    378
$1.00 - $1.50    359
$1.50 - $3.00    294
$0.50 - $5.00    293
$1.00 - $5.00    205
$1.00 - $7.00    182
$0.50 - $4.00    180
$1.50 - $5.00    157
$1.00 - $2.50    147
$0.50 - $7.00    136
Name: count, dtype: int64

 JUMP
RateRange
$1.5/H - $6/10H     799
$1/H - $4/10H       260
$1.5/H - $6/6H      163
$0.50/H - $5/13H     81
$1/H - $4/6H         71
$0.50/H - $5/14H     51
$1.5/H - $6/14H      39
$1.50/H - $6/10H     26
$1.5/H - $6/7H       11
$0.5/H - $2/6H        6
$0.50/H - $3/10H      1
$2/H - $8/10H         1
Name: count, dtype: int64


In [19]:
meters["price_numbers"] = meters["RateRange"].str.findall(r"\$(\d+(?:\.\d+)?)")

meters[["RateType", "RateRange", "price_numbers"]].head(15)

,RateType,RateRange,price_numbers
0,TOD,$5.00 - $7.00,"[5.00, 7.00]"
1,FLAT,$3.00,[3.00]
2,FLAT,$3.00,[3.00]
3,FLAT,$1.50,[1.50]
4,FLAT,$1.50,[1.50]
5,FLAT,$1.50,[1.50]
6,FLAT,$1.50,[1.50]
7,FLAT,$1.50,[1.50]
8,FLAT,$1.50,[1.50]
9,FLAT,$1.50,[1.50]


## 6. Construct the price proxy and numeric coordinates

I define `price_midpoint` as the arithmetic mean of the extracted dollar amounts: it equals a single stated price or the midpoint of two stated amounts. For TOD and JUMP schedules it is only a summary proxy, not the price every driver paid; in JUMP strings, the amounts can even refer to different durations, so this is not a normalized hourly price. I also separate `LatLng` into numeric `latitude` and `longitude`, while retaining `RateType` and `RateRange` to preserve the original pricing context.


In [20]:
meters["price_numbers"] = meters["RateRange"].str.findall(r"\$(\d+(?:\.\d+)?)")

meters["price_numbers"] = meters["price_numbers"].apply(
    lambda x: [float(i) for i in x]
)

meters["price_midpoint"] = meters["price_numbers"].apply(
    lambda x: sum(x) / len(x) if len(x) > 0 else None
)

meters[["latitude", "longitude"]] = (
    meters["LatLng"]
    .str.strip("()")
    .str.split(",", expand=True)
    .astype(float)
)

print(meters[["RateType", "RateRange", "price_midpoint"]].head(10))

print("\nPrice summary:")
print(meters["price_midpoint"].describe())

print("\nCoordinate summary:")
print(meters[["latitude", "longitude"]].describe())

  RateType      RateRange  price_midpoint
0      TOD  $5.00 - $7.00             6.0
1     FLAT          $3.00             3.0
2     FLAT          $3.00             3.0
3     FLAT          $1.50             1.5
4     FLAT          $1.50             1.5
5     FLAT          $1.50             1.5
6     FLAT          $1.50             1.5
7     FLAT          $1.50             1.5
8     FLAT          $1.50             1.5
9     FLAT          $1.50             1.5

Price summary:
count    35015.000000
mean         1.703784
std          0.809644
min          0.500000
25%          1.500000
50%          1.500000
75%          1.500000
max          8.000000
Name: price_midpoint, dtype: float64

Coordinate summary:
           latitude     longitude
count  35015.000000  35015.000000
mean      34.056893   -118.314527
std        1.301737      2.033623
min     -118.290443   -118.630592
25%       34.045335   -118.429528
50%       34.061636   -118.328036
75%       34.099362   -118.266839
max       34.207

## 7. Validate the numeric variables

I review the summaries above and check the constructed variables for missing values and implausible ranges, without applying another filter. The saved outputs show no missing numeric values, but the coordinate summary includes a latitude below −90 and a positive maximum longitude, suggesting some source coordinate pairs may be reversed. That issue remains unresolved here; a successful numeric conversion does not establish that every coordinate is valid.


In [21]:
print("Missing time limits:", meters["time_limit_hours"].isnull().sum())
print("Missing prices:", meters["price_midpoint"].isnull().sum())
print("Missing latitude:", meters["latitude"].isnull().sum())
print("Missing longitude:", meters["longitude"].isnull().sum())

print("\nTime limit summary:")
print(meters["time_limit_hours"].describe())

Missing time limits: 0
Missing prices: 0
Missing latitude: 0
Missing longitude: 0

Time limit summary:
count    35015.000000
mean         2.320520
std          1.820516
min          0.250000
25%          2.000000
50%          2.000000
75%          2.000000
max         14.000000
Name: time_limit_hours, dtype: float64


## 8. Check matches to the cleaned occupancy sample

I use a left merge on SpaceID to check how the 4,185-space occupancy sample from Notebook 01 matches to the inventory. The saved results show 4,139 matches (98.90%) and 46 unmatched spaces, which I inspect without dropping them from the occupancy sample. This is a linkage check, not the final cross-sectional merge or an additional restriction on the inventory.


In [22]:
occupancy_clean = pd.read_csv(
    "../../data/processed/space_occupancy_aug2025.csv"
)

matched = occupancy_clean.merge(
    meters,
    on="SpaceID",
    how="left",
    indicator=True
)

print(matched["_merge"].value_counts())

print("\nMatched percentage:")
print((matched["_merge"] == "both").mean() * 100)

_merge
both          4139
left_only       46
right_only       0
Name: count, dtype: int64

Matched percentage:
98.90083632019116


In [23]:
unmatched = matched[matched["_merge"] != "both"]

print("Unmatched spaces:", len(unmatched))

unmatched[["SpaceID", "occupancy_rate", "active_days"]].head(20)

Unmatched spaces: 46


,SpaceID,occupancy_rate,active_days
5,61117,0.501990,31
6,61118,0.521684,31
37,61351,0.650139,31
38,61352,0.682325,31
39,61353,0.677709,31
40,61354,0.698670,31
41,61355,0.652248,31
42,61356,0.664612,31
43,61357,0.645580,31
44,61358,0.700899,29


## 9. Save the cleaned meter inventory

I save all 35,015 unique inventory spaces with the selected original fields and the four numeric variables created here. The original `MeteredTimeLimit`, `RateType`, and `RateRange` remain in the output, while the temporary `price_numbers` list is not exported. This meter-level file will be used in the later cross-sectional merge; it is not restricted to the spaces matched in the check above.


In [24]:
meters_clean = meters[
    [
        "SpaceID",
        "BlockFace",
        "MeterType",
        "RateType",
        "RateRange",
        "MeteredTimeLimit",
        "price_midpoint",
        "time_limit_hours",
        "latitude",
        "longitude"
    ]
].copy()

meters_clean.to_csv(
    "../../data/processed/meter_inventory_clean.csv",
    index=False
)

meters_clean.head()

,SpaceID,BlockFace,MeterType,RateType,RateRange,MeteredTimeLimit,price_midpoint,time_limit_hours,latitude,longitude
0,CB3201,201 8TH ST,Single-Space,TOD,$5.00 - $7.00,2HR,6.0,2.0,34.041630,-118.251873
1,WV156,10900 LE CONTE AVE,Single-Space,FLAT,$3.00,2HR,3.0,2.0,34.063573,-118.447284
2,WV250,10900 LE CONTE AVE,Single-Space,FLAT,$3.00,2HR,3.0,2.0,34.063589,-118.446051
3,SH248,1000 HIGHLAND AVE,Single-Space,FLAT,$1.50,1HR,1.5,1.0,34.089736,-118.338477
4,SH5,6201 SANTA MONICA BLVD,Single-Space,FLAT,$1.50,1HR,1.5,1.0,34.090827,-118.325045


## Summary

Removing seven exact duplicate records leaves 35,015 unique meter-space observations. I created `time_limit_hours`, `price_midpoint`, `latitude`, and `longitude` as candidate explanatory variables for the later OLS analysis, while preserving the original time-limit and pricing fields. Latitude and longitude will also allow neighborhood characteristics to be attached in the next stage, subject to resolving the coordinate issue noted above.

The price proxy simplifies TOD and JUMP schedules and should not be interpreted as a precise transaction price or, for mixed-duration JUMP amounts, a uniform hourly rate. The inventory is also a later snapshot than the August 2025 occupancy data, so its recorded prices and rules may not exactly describe that month.
